# Lab 2. Titanic survival classification

Predict whether a passenger survived (`Survived = 1`).

- **Data:** the files of the course Kaggle competition, built from the [Titanic Dataset](https://www.kaggle.com/datasets/yasserh/titanic-dataset): `train.csv` (891 passengers with answers) and `test.csv` (1000 passengers without answers, scored on the leaderboard). Seven features: `Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Embarked`; `Id` only matches the rows.
- **Metric:** F1 for the class `Survived = 1`, required level **> 0.70**; accuracy, precision and recall as well.
- **Plan:** data → split → EDA → preprocessing → baseline and five models compared by CV → decision threshold → final test evaluation → feature importance → error analysis → Kaggle submission.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
pd.set_option("display.notebook_repr_html", False)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 72

SEED = 42
TARGET = "Survived"
NUMERIC = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
CATEGORICAL = ["Sex", "Embarked"]
FEATURES = NUMERIC + CATEGORICAL
REQUIRED_F1 = 0.70

## 1. Data

In [ ]:
data = pd.read_csv("data/train.csv")
kaggle_test = pd.read_csv("data/test.csv")
conflicts = (data.groupby(FEATURES, dropna=False)[TARGET].nunique() > 1).sum()
overlap = len(kaggle_test.merge(data[FEATURES].drop_duplicates()))
NO_FARE = [c for c in FEATURES if c != "Fare"]
overlap_no_fare = len(kaggle_test.merge(data[NO_FARE].drop_duplicates()))
print(f"train.csv: {len(data)} passengers, test.csv (Kaggle, no answers): {len(kaggle_test)}")
print(f"survived: {data[TARGET].mean():.1%}, Id is unique: {data['Id'].is_unique}")
print(f"rows that repeat another row in all features: {data.duplicated(FEATURES).sum()}")
print(f"groups of identical rows with different outcomes: {conflicts}")
print(f"test.csv rows identical to a train.csv row: {overlap}, ignoring Fare: {overlap_no_fare}")
display(
    pd.DataFrame(
        {
            "dtype": data.dtypes.astype(str),
            "missing": data.isna().sum(),
            "missing in test.csv": kaggle_test.isna().sum(),
            "unique": data.nunique(),
        }
    ).T
)
data[NUMERIC].describe().T[["mean", "std", "min", "50%", "max"]]

**Findings**

- **Features:** `Pclass` is the ticket class (1–3), `SibSp` the number of siblings and spouses aboard, `Parch` the number of parents and children, `Fare` the ticket price, `Embarked` the port (C = Cherbourg, Q = Queenstown, S = Southampton).
- **Target:** 38% survived. The classes are imbalanced, so accuracy alone is misleading: "everybody died" already gives 62%.
- **Missing values:**
  - `Age`: 177 (20%) in train and 161 in `test.csv`. Imputed with the median plus an "age was missing" indicator. For age the mean and the median are close (29.7 and 28); the median is chosen because it depends less on extreme values. The fact that the age is unknown matters by itself: 28% of such passengers in train survived vs 38% overall (section 3).
  - `Embarked`: 2, imputed with the most frequent port (Southampton); for two rows this changes almost nothing. `test.csv` has none.
- **Identifying columns:** `Id` is unique for every row, so it can only help a model memorize passengers. It is not used as a feature (section 9 checks what happens if it is). `Name`, `Ticket` and `Cabin` were already removed by the organizers.
- **Duplicates:** 127 rows repeat another row in all seven features. These are simply different people with identical features: the largest group is 13 third-class men from Southampton with unknown age and a fare of 7.90. In 16 groups the outcomes differ. The rows are kept, because such passengers are real and frequent. None of the 1000 `test.csv` rows repeats a training row exactly, so remembering exact rows would not help on the leaderboard anyway. Ignoring `Fare`, 451 rows match; the fares in `test.csv` seem slightly noisy (values like 9.1133 or 8.9682).
- **Need processing:** `Sex` and `Embarked` are categorical (one-hot encoding). `Fare` has a long tail (median 14.5, maximum 512, 15 zero fares). `Pclass` is ordinal and stays a number.

## 2. Train/test split

- **80/20** (712 / 179) with `stratify=Survived`, so both parts keep 38% survivors. `random_state=42`.
- All models use this one split. Imputers, encoders and scalers live inside a `Pipeline`, so they are fitted on the training data only, inside every CV fold as well.
- The model and the decision threshold are chosen by 5-fold stratified CV on train. The test set is used once, in section 7.

In [ ]:
train, test = train_test_split(data, test_size=0.2, stratify=data[TARGET], random_state=SEED)
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"train: {len(train)}, survived {y_train.mean():.1%}")
print(f"test:  {len(test)}, survived {y_test.mean():.1%}")

## 3. Exploratory analysis

On the training part only.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, column in zip(axes.flat, ["Sex", "Pclass", "Embarked", "SibSp", "Parch"], strict=False):
    sns.barplot(data=train, x=column, y=TARGET, errorbar=None, color="#4393c3", ax=ax)
    ax.set_ylabel("survival rate")
sns.histplot(
    data=train,
    x="Age",
    hue=TARGET,
    bins=30,
    multiple="stack",
    palette={0: "#d6604d", 1: "#4393c3"},
    ax=axes[1, 2],
)
axes[1, 2].set_ylabel("passengers")
axes[1, 2].get_legend().set_title("survived")
fig.tight_layout()
plt.show()

age_group = pd.cut(train["Age"], [0, 12, 60, 80], labels=["0-12", "13-60", "61-80"])
display(train.pivot_table(index="Sex", columns="Pclass", values=TARGET, aggfunc="mean"))
display(train.groupby(age_group.astype(str), sort=False)[TARGET].agg(["mean", "size"]).T)
age_known = train["Age"].isna().map({False: "age known", True: "age missing"})
display(train.groupby(age_known)[TARGET].agg(["mean", "size"]).T)
train.groupby("Pclass")["Fare"].median().to_frame("median Fare").T

**Findings**

- **Sex is the strongest factor:** 73% of women vs 19% of men survived ("women and children first").
- **Class:** 62% / 49% / 23% for 1st / 2nd / 3rd class. Sex and class interact: women of the 1st and 2nd class almost all survived (96%, 92%), women of the 3rd class only 47%; men of the 1st class 37%, of the 2nd and 3rd 16% and 13%.
- **Age:** children aged 12 or younger survived 55% of the time (58 children), passengers over 60 only 28% (but there are just 18 of them), the rest 40%. Passengers with a missing age survived 28% of the time (145 of them).
- **Family:** one or two relatives aboard is better than none (about 50% vs 35% for `SibSp`), while large families died: with 4 or more siblings almost nobody survived. The effect is non-linear.
- **Embarked:** Cherbourg 56% vs Southampton 33%. This is mostly class: half of the Cherbourg passengers travelled in 1st class, compared with a fifth in Southampton. The port is not a reason by itself.
- `Fare` is another form of class (median 66.6 for 1st class vs 8.05 for 3rd).

## 4. Preprocessing and models

| Step | Numeric: `Pclass`, `Age`, `SibSp`, `Parch`, `Fare` | Categorical: `Sex`, `Embarked` |
| --- | --- | --- |
| Missing values | median + "age was missing" indicator | most frequent value |
| Encoding | none | one-hot |
| Scaling | standardization, only for SVM and KNN | not needed: one-hot columns are already 0/1 |

`Pclass` is kept as a number because it is **ordinal** (1st > 2nd > 3rd). Trees split it with a threshold anyway, and for SVM/KNN the distance between classes 1 and 3 being larger than between 1 and 2 makes sense.

**Where scaling matters:**

- **SVM and KNN** work with distances between passengers. Unscaled, `Fare` (0–512) would dominate `Pclass` (1–3) and the 0/1 columns, and the model would effectively look only at the fare. After standardization every numeric feature has mean 0 and standard deviation 1.
- **For KNN it is especially important**, because the neighbors are chosen directly by distance. Unscaled, the "nearest" passengers would simply be those with a similar fare, and sex and class would hardly matter. In SVM the distance enters the RBF kernel, so it needs scaling too.
- **Trees** compare one feature with a threshold at a time, so the scale does not change them.

**Models** use default hyperparameters (tuning is the topic of lab 3):

- **Decision Tree:** a sequence of if/else splits, grown until the leaves are pure.
- **Random Forest:** 500 trees on bootstrap samples; at every split a tree chooses among a random subset of features; majority vote.
- **Gradient Boosting:** 100 shallow trees (depth 3), each correcting the errors of the previous ones.
- **SVM:** the widest margin between the classes, with an RBF kernel for a non-linear boundary.
- **KNN:** the majority class among the 5 nearest passengers.
- **Baseline:** `DummyClassifier`, always the most frequent class (died).

In [ ]:
def preprocessor(scale: bool, numeric: list[str] = NUMERIC) -> ColumnTransformer:
    numeric_steps = [SimpleImputer(strategy="median", add_indicator=True)]
    if scale:
        numeric_steps.append(StandardScaler())
    return ColumnTransformer(
        [
            ("num", make_pipeline(*numeric_steps), numeric),
            (
                "cat",
                make_pipeline(
                    SimpleImputer(strategy="most_frequent"),
                    OneHotEncoder(handle_unknown="ignore"),
                ),
                CATEGORICAL,
            ),
        ],
        verbose_feature_names_out=False,
    )


models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Decision Tree": make_pipeline(preprocessor(False), DecisionTreeClassifier(random_state=SEED)),
    "Random Forest": make_pipeline(
        preprocessor(False), RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=SEED)
    ),
    "Gradient Boosting": make_pipeline(
        preprocessor(False), GradientBoostingClassifier(random_state=SEED)
    ),
    "SVM": make_pipeline(preprocessor(True), SVC(random_state=SEED)),
    "KNN": make_pipeline(preprocessor(True), KNeighborsClassifier()),
}

## 5. Model selection by cross-validation

- **Accuracy:** the share of correct answers.
- **Precision:** the share of passengers predicted as survivors who really survived.
- **Recall:** the share of real survivors that the model found.
- **F1:** the harmonic mean of precision and recall. It is high only when both are high, so the trivial "everybody died" gets 0.

In the table, train F1 is the mean F1 on the training parts of the folds, CV F1 on the validation parts, and CV std is the spread of CV F1 over the folds.

In [ ]:
cv_rows = {}
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=CV, scoring="f1", return_train_score=True)
    cv_rows[name] = {
        "train F1": scores["train_score"].mean(),
        "CV F1": scores["test_score"].mean(),
        "CV std": scores["test_score"].std(),
    }
cv_table = pd.DataFrame(cv_rows).T
cv_table

**Findings**

- **SVM is the best by CV (0.736)** and the most even on these five folds: the std of its F1 is only 0.008, the others have 0.03–0.06.
- **Decision Tree overfits:** F1 0.98 on the training folds and 0.67 on the validation folds. An unrestricted tree grows until it memorizes almost every training passenger, and this does not transfer to new ones. It has the lowest CV and is the only model below 0.70 by CV.
- **Random Forest** has the same train F1 (0.98, its trees are also deep), but averaging 500 trees raises CV to 0.711.
- **Gradient Boosting** (0.716) overfits less (train 0.87) because its trees are shallow.
- **KNN** (0.701) is just above the required level: a vote of 5 neighbors is noisy.
- **Baseline** never predicts a survivor, so its F1 is 0.

## 6. Decision threshold

SVM gives every passenger a **score**, the value of its decision function: the sign shows the side of the boundary and the magnitude how far from it the passenger is, and by default predicts "survived" when the score is above 0. Moving this threshold trades precision for recall. The threshold is chosen on **out-of-fold** predictions: each training passenger is scored by the model of the fold in which that passenger was in the validation part, so no score comes from a model that saw the passenger. The test set is not used.

In [ ]:
def score(model: object, features: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "decision_function"):
        return model.decision_function(features)
    return model.predict_proba(features)[:, 1] - 0.5


best_name = cv_table.drop(index="Baseline")["CV F1"].idxmax()
best = models[best_name]
oof = np.zeros(len(y_train))
for fit_idx, val_idx in CV.split(X_train, y_train):
    fold_model = clone(best).fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
    oof[val_idx] = score(fold_model, X_train.iloc[val_idx])

thresholds = np.round(np.arange(-1.0, 1.01, 0.1), 1)
curve = pd.DataFrame(
    {
        "precision": [precision_score(y_train, oof > t, zero_division=0) for t in thresholds],
        "recall": [recall_score(y_train, oof > t) for t in thresholds],
        "F1": [f1_score(y_train, oof > t) for t in thresholds],
    },
    index=thresholds,
)
best_threshold = curve["F1"].idxmax()

fig, ax = plt.subplots(figsize=(8, 3.5))
curve.plot(ax=ax, color=["#8073ac", "#5aae61", "#d6604d"], marker="o", markersize=3)
ax.axvline(0, color="grey", ls=":", label="default threshold")
ax.axvline(best_threshold, color="#d6604d", ls="--", label="chosen threshold")
ax.set_xlabel(f"threshold on the {best_name} score")
ax.set_title(f"{best_name}: out-of-fold predictions on train")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()
print(f"best model by CV: {best_name}")
print(f"out-of-fold F1 at the default threshold 0: {curve.loc[0.0, 'F1']:.3f}")
print(f"out-of-fold F1 at the chosen threshold {best_threshold:+.1f}: {curve['F1'].max():.3f}")

**Findings**

- A lower threshold means more passengers predicted as survivors: recall grows and precision falls. F1 is highest at **−0.2** (0.748 vs 0.736 at the default 0).
- The default SVM is cautious: it makes fewer false "survived" than missed survivors, and F1, which looks at the survivors, gains from a little more recall.
- The gain is small, and the F1 curve is almost flat between −0.6 and 0, so the exact value is not critical. The threshold is fixed now, before the test.
- 0.748 is slightly optimistic, because the threshold was chosen on these same out-of-fold predictions. The fair comparison of the five models under equal conditions is the table in section 5, where all of them use the default threshold. The threshold is tuned after the model is chosen and only for it.

## 7. Final evaluation on the test set

In [ ]:
def metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
    }


predictions, train_predictions = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    train_predictions[name] = model.predict(X_train)
final_name = f"{best_name}, threshold {best_threshold:+.1f}"
predictions[final_name] = (score(best, X_test) > best_threshold).astype(int)
train_predictions[final_name] = (score(best, X_train) > best_threshold).astype(int)

results = pd.DataFrame({name: metrics(y_test, y_pred) for name, y_pred in predictions.items()}).T
results.insert(0, "train F1", [f1_score(y_train, y_pred) for y_pred in train_predictions.values()])
results.insert(1, "CV F1", [*cv_table["CV F1"], curve["F1"].max()])
results[f"F1 > {REQUIRED_F1}"] = results["F1"] > REQUIRED_F1
results

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for ax, (name, y_pred) in zip(axes.flat, predictions.items(), strict=False):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=["died", "survived"], colorbar=False, cmap="Blues", ax=ax
    )
    ax.set_title(name)
    ax.grid(False)
axes.flat[-1].axis("off")
fig.tight_layout()
plt.show()

**Findings: metrics**

- **The baseline has 61.5% accuracy but F1 = 0**, which is why accuracy is not the main metric.
- **All five models pass F1 > 0.70 on the test.** The final model (SVM with threshold −0.2) gets **F1 0.803**, accuracy 0.866. For the tree (CV 0.670) and KNN (CV 0.701) this is a lucky test rather than a safe margin.
- **Every model scores 0.05–0.08 higher on the test than by CV.** The final models are trained on all 712 training passengers instead of 570 in CV, and the test part is somewhat easier: for example, 61% of its 3rd class women survived vs 47% in train (section 9).
- The test has only 179 passengers (69 survivors), and one passenger changes F1 by about 0.01, so the exact order on it is noise: boosting is slightly above the default SVM (0.797 vs 0.793), while by CV it is a little worse (0.716 vs 0.736). The choice is made by CV.
- **Boosting vs forest:** almost equal by CV (0.716 vs 0.711). On the test boosting has a higher F1 (0.797 vs 0.782) and a different error profile: precision 0.96 with recall 0.68, while the forest finds more survivors (recall 0.75) at the cost of more FP.
- **The tree's overfitting is confirmed:** train 0.98 vs test 0.746, the lowest of the five.

**Findings: confusion matrices** (test: 110 died, 69 survived)

| Model | FP (died, predicted survived) | FN (survived, predicted died) |
| --- | --- | --- |
| Decision Tree | 15 | 19 |
| Random Forest | 12 | 17 |
| Gradient Boosting | 2 | 22 |
| SVM | 4 | 21 |
| KNN | 8 | 22 |
| SVM, threshold −0.2 | 4 | 20 |

- Every model makes more FN than FP. Boosting and SVM are the most precise (precision 0.96 and 0.92) but miss about 30% of the survivors. The deep trees find more survivors but pay with many more FP (12–15).
- **FP vs FN in this task:**
  - FP: the model says "survived" for someone who died;
  - FN: it misses a survivor.

  If the model were used, for example, to plan a search for survivors, FN would be worse: a person who may be alive would not be looked for, and the threshold would be moved further down. For an overall estimate both errors matter, which is what F1 balances.

## 8. Feature importance

In [ ]:
TREES = ["Decision Tree", "Random Forest", "Gradient Boosting"]
importance = pd.DataFrame(
    {
        name: pd.Series(
            models[name][-1].feature_importances_, index=models[name][0].get_feature_names_out()
        )
        for name in TREES
    }
).sort_values("Random Forest", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, name, color in zip(axes, TREES, ["#f4a582", "#5aae61", "#8073ac"], strict=True):
    importance[name].iloc[::-1].plot.barh(ax=ax, color=color)
    ax.set_title(name)
fig.tight_layout()
plt.show()


def source_feature(column: str) -> str:
    return "Age" if column.startswith("missingindicator") else column.split("_")[0]


display(importance.round(3))
importance.groupby(source_feature).sum().sort_values("Random Forest", ascending=False).round(3)

Importance here is `feature_importances_` (MDI): the total decrease of impurity in the nodes where a tree splits on the feature.

**Findings**

1. **Grouped by original feature** (last table), all three models put **sex first** (DT 0.30, RF 0.27, GB 0.45), then fare, age and class in different order. `SibSp`, `Parch` and `Embarked` are minor.
2. **DT vs RF vs GB:**
   - The **Decision Tree** uses `Sex_female` (0.30) and gives `Sex_male` exactly 0: it is the same information, and one column is enough. Then come `Age` (0.26) and `Fare` (0.24).
   - **Random Forest** spreads the importance between `Sex_male` and `Sex_female`, because at every split only a random subset of features is available, sometimes with one sex column and sometimes with the other. Its top column is `Fare` (0.26).
   - **Gradient Boosting** relies on sex most clearly (0.45), then on `Fare` and `Pclass` (0.17 each).
3. **`Age` and `Fare` are overrated by the deep trees.** Continuous features offer many split points, and deep trees use them to separate individual passengers. Impurity importance counts these splits too. Boosting, with shallow trees, ranks them lower and gives class more weight.
4. **Against EDA:** sex is first, as expected. Class is the second factor in EDA, and the models get it partly through `Fare`, its proxy. `Embarked` is almost unused: its effect in EDA was explained by class.

**Is an important feature the cause of survival? No.** Importance shows what the model uses to predict, not what causes survival.

- `Fare` did not save anyone. It is a marker of class, and class means cabin location and access to the boats.
- Even for sex the cause is not biology but the evacuation rule "women and children first".

Besides, the data comes from observation, not from an experiment, and correlated features (fare and class, the two sex columns) share or take over each other's importance.

## 9. Error analysis

In [ ]:
errors = test.assign(predicted=predictions[final_name], score=score(best, X_test))
errors["error"] = "correct"
errors.loc[(errors[TARGET] == 0) & (errors["predicted"] == 1), "error"] = "FP"
errors.loc[(errors[TARGET] == 1) & (errors["predicted"] == 0), "error"] = "FN"
print(f"model: {final_name}")
display(test.pivot_table(index="Sex", columns="Pclass", values=TARGET, aggfunc="mean"))
pd.crosstab([errors["Sex"], errors["Pclass"]], errors["error"], margins=True)

In [ ]:
wrong = errors[errors["error"] != "correct"].sort_values("score")
wrong.iloc[[0, 1, 2, -3, -2, -1]][["Id", *FEATURES, TARGET, "score"]]

**Findings** (SVM with threshold −0.2: 20 FN, 4 FP)

- **Men are almost always predicted as dead.** 16 of the 20 FN are men: 5 of 16 men in 1st class, 1 of 28 in 2nd, 10 of 76 in 3rd. The model has essentially learned "man → died". It is right for 104 of 120 men, but it finds only 6 of the 22 male survivors.
- **By error rate the hardest groups are 1st class men** (5 of 16, all FN) **and 3rd class women** (7 of 28, both kinds: 4 FN, 3 FP). In train 47% of 3rd class women survived, close to a coin flip; in the test more of them survived (61%, first table), so FN outnumber FP among them.
- **1st and 2nd class women** are predicted almost perfectly (1 error out of 31).
- **Examples** (`score` is the SVM score; below −0.2 means "died"):
  - Id 54109971: a 17-year-old girl in 3rd class travelling with six relatives survived. The model was very sure she died (−1.46), because large 3rd class families almost never survived.
  - Ids 10000115 and 10000246: young 3rd class men who survived. Every feature says "died"; this is luck that the features cannot explain.
  - Id 10000257: a 57-year-old 2nd class woman who died, while the model was confident she survived (0.68).
  - Ids 10000199 and 65249351: 3rd class women right at the boundary (0.05 and 0.08), exactly the coin-flip group.

**Check for identifying features.** The same Random Forest is trained with `Id` added as a feature, and SVM is trained without `Fare`, which can identify a ticket group (relatives on one ticket paid the same fare).

In [ ]:
with_id = make_pipeline(
    preprocessor(False, [*NUMERIC, "Id"]),
    RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=SEED),
)
id_cv = cross_validate(with_id, train[[*FEATURES, "Id"]], y_train, cv=CV, scoring="f1")
with_id.fit(train[[*FEATURES, "Id"]], y_train)
id_importance = pd.Series(
    with_id[-1].feature_importances_, index=with_id[0].get_feature_names_out()
).sort_values(ascending=False)

no_fare = make_pipeline(
    preprocessor(True, [c for c in NUMERIC if c != "Fare"]), SVC(random_state=SEED)
)
no_fare_cv = cross_validate(no_fare, X_train, y_train, cv=CV, scoring="f1")

sequential = train["Id"] < 10_001_000
print(f"correlation of Id with {TARGET} on train: {train['Id'].corr(y_train):.3f}")
print(f"survived with sequential Id ({sequential.sum()} rows): {y_train[sequential].mean():.1%}")
print(f"survived with random Id: {y_train[~sequential].mean():.1%}")
print(f"Random Forest CV F1 without Id: {cv_table.loc['Random Forest', 'CV F1']:.3f}")
print(f"Random Forest CV F1 with Id:    {id_cv['test_score'].mean():.3f}")
print(f"rank of Id in its importance:   {id_importance.index.get_loc('Id') + 1}")
print(f"SVM CV F1 with Fare:    {cv_table.loc['SVM', 'CV F1']:.3f}")
print(f"SVM CV F1 without Fare: {no_fare_cv['test_score'].mean():.3f}")
id_importance.head(5).round(3).to_frame("importance with Id").T

**Findings**

- `Id` has no linear relation to survival (correlation on train −0.06). Some rows have sequential Ids (10000000, 10000001 and so on), probably the original row order, but their survival rate is almost the same as for the rest (40% vs 38%). Yet `Id` becomes the **2nd most important feature** of the forest (0.18), almost equal to `Fare`. The deep trees use any unique number to split off individual passengers, which is memorization. CV F1 moves from 0.711 to 0.722, a change well inside the fold spread (std 0.049), so it is noise, not information. Impurity importance does not protect against such features, which is why `Id` is excluded.
- Without `Fare`, SVM loses only 0.007 of CV F1 (0.736 → 0.729). Its quality comes from general rules (sex, class, age), not from recognizing ticket groups; the fare adds a little as extra information about class.

## 10. Kaggle submission

After the evaluation the chosen solution is fixed: SVM with default parameters and threshold −0.2. For the submission the same pipeline is trained again on all 891 passengers of `train.csv`: the 179 test passengers add 25% more data, and nothing is chosen anymore. The predictions for `test.csv` are saved to `submission.csv` (`Id,Survived`, 1000 rows) and uploaded to the competition. The leaderboard score is not used for any decision.

In [ ]:
final_model = clone(best).fit(data[FEATURES], data[TARGET])
submission = pd.DataFrame(
    {
        "Id": kaggle_test["Id"],
        TARGET: (score(final_model, kaggle_test[FEATURES]) > best_threshold).astype(int),
    }
)
submission.to_csv("submission.csv", index=False)
print(f"rows: {len(submission)}, predicted survivors: {submission[TARGET].mean():.1%}")
submission.head()

The model predicts 34% survivors in `test.csv` against 38% in train: as on the test part, it is cautious and misses some survivors rather than inventing them.

## Conclusions

| Model | CV F1 | Test accuracy | Test precision | Test recall | Test F1 |
| --- | --- | --- | --- | --- | --- |
| Baseline | 0.000 | 0.615 | 0.000 | 0.000 | 0.000 |
| Decision Tree | 0.670 | 0.810 | 0.769 | 0.725 | 0.746 |
| Random Forest | 0.711 | 0.838 | 0.812 | 0.754 | 0.782 |
| Gradient Boosting | 0.716 | 0.866 | 0.959 | 0.681 | 0.797 |
| SVM | 0.736 | 0.860 | 0.923 | 0.696 | 0.793 |
| KNN | 0.701 | 0.832 | 0.855 | 0.681 | 0.758 |
| **SVM, threshold −0.2** | **0.748** | **0.866** | 0.925 | **0.710** | **0.803** |

The CV F1 of the SVM with the tuned threshold is slightly optimistic, because the threshold was chosen on the same out-of-fold predictions.

- All five models pass **F1 > 0.70** on the test; by CV the tree is below it. The final model, chosen by CV, is **SVM with threshold −0.2 (F1 0.803)**; SVM was also the most even across the folds.
- The main factors are sex, then class (and fare as its proxy) and age.
- The unrestricted tree memorizes the training set (train F1 0.98) and is the weakest; averaging (forest) and shallow trees (boosting) reduce overfitting.
- The errors are mostly missed survivors among men and mistakes on 3rd class women, i.e. where survival was close to random.
- `Id` is excluded: the forest uses it heavily although it carries no information, which is memorization.